# Topic 7 — Decision Trees

> **Goal of this notebook:** understand how a decision tree splits data with simple if/else questions, the impurity measures (**Gini**, **entropy**) that pick the best splits, and how to stop it from overfitting. We'll visualise a real tree and compute information gain by hand.

**Contents**
1. Concept & intuition
2. The mathematics (Gini, entropy, information gain)
3. Overfitting & pruning
4. The dataset: Iris
5. Training & visualising the tree
6. Feature importance
7. Controlling depth (train vs test)
8. Information gain from scratch
9. Pros, cons & when to use
10. Summary

## 1. Concept & Intuition

A decision tree asks a sequence of yes/no questions about the features to partition the data into increasingly **pure** groups. Each **internal node** is a test (e.g. *petal length ≤ 2.5?*), each **branch** an answer, and each **leaf** a prediction.

It's grown **greedily and recursively**: at every node the algorithm picks the split that best separates the classes, then repeats on each child until the leaves are pure or a stopping rule kicks in. The result is a flowchart a human can read — trees are the most **interpretable** model.

## 2. The Mathematics

We measure how mixed a node is with an **impurity** score (for classes with proportions $p_k$):

**Gini impurity:** $\;G = 1 - \sum_{k} p_k^2$

**Entropy:** $\;H = -\sum_{k} p_k \log_2 p_k$

Both are 0 when a node is pure (one class) and maximal when classes are evenly mixed. A split is scored by **information gain** — the drop in impurity, weighted by child sizes:

$$\text{Gain} = I(\text{parent}) - \sum_{j} \frac{n_j}{n}\, I(\text{child}_j)$$

The algorithm tries every feature and threshold and keeps the split with the **highest gain**.

## 3. Overfitting & Pruning

A fully grown tree memorises the training data (every leaf pure) and generalises poorly. We control complexity with:

- **max_depth** — limit how deep the tree grows.
- **min_samples_leaf / min_samples_split** — require enough samples to split.
- **ccp_alpha** — cost-complexity pruning that removes weak branches.

These are the tree's regularisation knobs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. Training & Visualising the Tree

In [ ]:
tree = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree (max_depth=3)')
plt.show()

## 6. Feature Importance

Trees rank features by how much each reduces impurity across all its splits.

In [ ]:
imp = tree.feature_importances_
order = np.argsort(imp)
plt.barh([iris.feature_names[i] for i in order], imp[order])
plt.xlabel('importance'); plt.title('Feature importance'); plt.show()

## 7. Controlling Depth (Train vs Test)

Deeper trees fit training data perfectly but overfit — watch the gap widen.

In [ ]:
depths = range(1, 11)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))

plt.plot(list(depths), train_acc, marker='o', label='train')
plt.plot(list(depths), test_acc, marker='s', label='test')
plt.xlabel('max_depth'); plt.ylabel('accuracy')
plt.title('Overfitting as the tree deepens'); plt.legend(); plt.show()

## 8. Information Gain From Scratch

Let's reproduce how the tree picks its very first split — by finding the feature/threshold with the highest Gini information gain.

In [ ]:
def gini(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1 - (p ** 2).sum()

def best_split(X, y):
    parent = gini(y)
    best = (None, None, -1)  # feature, threshold, gain
    for f in range(X.shape[1]):
        for thr in np.unique(X[:, f]):
            left = y[X[:, f] <= thr]
            right = y[X[:, f] > thr]
            if len(left) == 0 or len(right) == 0:
                continue
            w = (len(left) * gini(left) + len(right) * gini(right)) / len(y)
            gain = parent - w
            if gain > best[2]:
                best = (f, thr, gain)
    return best

f, thr, gain = best_split(X_train, y_train)
print(f'Best first split: "{iris.feature_names[f]}" <= {thr:.2f}  (info gain={gain:.3f})')
print('scikit-learn root split feature:', iris.feature_names[tree.tree_.feature[0]],
      '<=', round(tree.tree_.threshold[0], 2))

Our hand-computed best split matches the feature scikit-learn chose at the root — confirming we understand how the tree grows.

## 9. Pros, Cons & When to Use

**Pros**
- Highly **interpretable** — you can read the rules.
- Handles numeric & categorical features, needs no scaling.
- Captures non-linear relationships and feature interactions.

**Cons**
- **Overfits** easily without pruning.
- **Unstable** — small data changes can yield a very different tree (high variance).
- A single tree is rarely the most accurate model.

**When to use**
- When interpretability/explanation is the priority.
- As the building block for ensembles (Random Forest, Boosting — the next topics).

## 10. Summary

- Decision trees split data greedily to maximise **information gain** (reduce **Gini**/**entropy**).
- Fully grown trees overfit; **max_depth**, **min_samples_leaf**, and **ccp_alpha** prune them.
- They are the most interpretable model and need no feature scaling.
- Our from-scratch split search reproduced scikit-learn's root split, and the depth experiment made overfitting visible.